# Notebook 2 — ChromaDB

Notebook 1 built a similarity search in a few lines: score every vector, sort, keep the top
k. That is enough for six sentences. A real app needs more: the vectors must survive a
restart, the search must be able to filter ("only the IT FAQ", "only 2026"), documents
change, and the search must stay fast as the data grows. A **vector database** does all of
this. This notebook uses **ChromaDB**, an open-source vector database that runs inside your
Python program, with no server to set up.

**What you will learn**

1. Clients, collections, and records.
2. How to add documents with ids and metadata.
3. How to query, and what Chroma's distances mean.
4. Metadata filters and text filters.
5. How to update and delete records.
6. How to save a collection to disk and load it again.
7. How to use Chroma in a LangChain RAG chain.

**Before you start**

```bash
ollama pull nomic-embed-text   # embedding model
ollama pull gpt-oss:20b        # chat model, only for the last section
uv sync
```

Then choose the `.venv` Python kernel for this notebook. ⏱️ About 45 minutes.

## 1. Setup

In [1]:
import json
import shutil
import time
from pathlib import Path

import chromadb
import numpy as np
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

from roadmap import get_chat_model, get_embeddings, get_settings
from roadmap.loaders import load_html, load_text

settings = get_settings()
DATA = Path("data")
STAGE2_DATA = Path("../02-rag-fundamentals/data")
DB_PATH = Path("chroma_db")  # listed in .gitignore, so it is never committed
print("chromadb", chromadb.__version__)

chromadb 1.5.9


## 2. Clients, collections, and records

![Inside a Chroma collection](assets/chroma_collection.svg)

| Part | What it is |
|---|---|
| **Client** | Your connection to Chroma. `EphemeralClient` keeps everything in memory; `PersistentClient` saves to a folder. |
| **Collection** | A named set of records, like a table. It has one embedding function and one distance measure. |
| **Record** | One stored item: a unique **id**, the **document** (the text), its **embedding**, and **metadata** (facts such as source, team, or year). |

Chroma can call Ollama itself: `OllamaEmbeddingFunction` embeds every document you add
and every question you ask, with the same `nomic-embed-text` model as before.

In [2]:
embedding_function = OllamaEmbeddingFunction(
    url=settings.ollama_host, model_name=settings.embed_model
)

client = chromadb.EphemeralClient()  # in memory, for trying things out
collection = client.create_collection(
    name="brightleaf",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},  # the distance measure, see section 4
)
print(f"Collection {collection.name!r}: {collection.count()} records")

Collection 'brightleaf': 0 records


## 3. Add documents

Our data is Brightleaf's knowledge base:

- the employee handbook and the IT FAQ from Stage 2, split at their headings,
- and a new file, [`data/announcements.json`](data/announcements.json): twelve made-up
  company announcements from 2024 to 2026, each with a date and a team.

Every record needs a unique id, the text, and metadata. Metadata values must be simple:
strings, numbers, or booleans. We store the year as a number, so we can later filter with
"2026 or later".

In [3]:
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "title"), ("##", "section")], strip_headers=False
)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

records = []
for doc in load_text(STAGE2_DATA / "handbook.md") + load_html(STAGE2_DATA / "it_faq.html"):
    source = doc.metadata["source"]
    number = 0
    for part in header_splitter.split_text(doc.page_content):
        section = part.metadata.get("section") or part.metadata.get("title", "")
        for chunk in size_splitter.split_text(part.page_content):
            number += 1
            records.append(
                {
                    "id": f"{Path(source).stem}-{number:02d}",  # handbook-01, it_faq-01, ...
                    "document": chunk,
                    "metadata": {"source": source, "section": section},
                }
            )

announcements = json.loads((DATA / "announcements.json").read_text(encoding="utf-8"))
for item in announcements["announcements"]:
    records.append(
        {
            "id": item["id"],
            "document": f"{item['title']}. {item['text']}",
            "metadata": {
                "source": "announcements",
                "section": item["title"],
                "team": item["team"],
                "date": item["date"],
                "year": int(item["date"][:4]),
            },
        }
    )

print(len(records), "records, for example:")
for record in (records[0], records[-1]):
    print(" ", record["id"], record["metadata"])

36 records, for example:
  handbook-01 {'source': 'handbook.md', 'section': 'Brightleaf Employee Handbook'}
  news-12 {'source': 'announcements', 'section': 'Book your learning days', 'team': 'People', 'date': '2026-09-14', 'year': 2026}


`add()` takes three parallel lists. Chroma calls the embedding function for every
document and stores the result next to it:

In [4]:
start = time.perf_counter()
collection.add(
    ids=[r["id"] for r in records],
    documents=[r["document"] for r in records],
    metadatas=[r["metadata"] for r in records],
)
print(f"Added {collection.count()} records in {time.perf_counter() - start:.1f} s\n")

stored = collection.get(ids=["news-09"], include=["documents", "metadatas", "embeddings"])
print(stored["documents"][0])
print(stored["metadatas"][0])
print(
    "embedding:",
    len(stored["embeddings"][0]),
    "numbers, starting with",
    np.round(stored["embeddings"][0][:4], 3),
)

Added 36 records in 0.7 s

New VPN app. A new VPN app replaces the old one. Install it from the software center before 31 March 2026; after that date, the old app stops working.
{'section': 'New VPN app', 'source': 'announcements', 'year': 2026, 'team': 'IT', 'date': '2026-02-16'}
embedding: 768 numbers, starting with [ 0.022 -0.029 -0.141  0.017]


## 4. Query

`query()` embeds the question and returns the closest records. It accepts a **list** of
questions, so every field in the result is a list with one inner list per question. A
small helper prints the first one:

In [5]:
def show(results: dict) -> None:
    """Print one line per result of the first question: distance, id, and section."""
    rows = zip(results["ids"][0], results["distances"][0], results["metadatas"][0], strict=True)
    for record_id, distance, metadata in rows:
        print(f"   {distance:.3f}  {record_id:<13} {metadata['section']}")


for question in [
    "Is the office open between Christmas and New Year?",
    "What changed with the VPN?",
    "Where can I park my bike?",
]:
    print(question)
    show(collection.query(query_texts=[question], n_results=3))

Is the office open between Christmas and New Year?
   0.157  news-07       Office closed over the holidays
   0.379  handbook-03   2. Remote work
   0.384  handbook-05   3. Vacation
What changed with the VPN?
   0.321  news-09       New VPN app
   0.364  it_faq-03     How do I connect to the VPN?
   0.460  it_faq-04     What is the guest Wi-Fi password?
Where can I park my bike?
   0.253  news-10       Bike parking in the basement
   0.533  it_faq-05     How do I get new software?
   0.539  it_faq-02     How do I reset my password?


The announcements are found right away, and the VPN question finds the new announcement
and the IT FAQ answer together.

The numbers are **distances**: smaller is closer. Which distance exactly? Let's check
with the math from Notebook 1, using the first result of the Christmas question:

In [6]:
question_vector = np.array(
    embedding_function(["Is the office open between Christmas and New Year?"])[0]
)
record_vector = np.array(collection.get(ids=["news-07"], include=["embeddings"])["embeddings"][0])

cosine = float(question_vector @ record_vector)  # both have length 1
print(f"cosine similarity: {cosine:.3f}")
print(f"1 - cosine:        {1 - cosine:.3f}   <- the distance Chroma returned")

cosine similarity: 0.843
1 - cosine:        0.157   <- the distance Chroma returned


With `"space": "cosine"`, Chroma returns the **cosine distance**, 1 − cosine similarity.
The setting has three options. Let's run the same question in all three:

In [7]:
news = [r for r in records if r["metadata"]["source"] == "announcements"]

for space in ("cosine", "ip", "l2"):
    test = client.create_collection(
        f"test-{space}",
        embedding_function=embedding_function,
        configuration={"hnsw": {"space": space}},
    )
    test.add(ids=[r["id"] for r in news], documents=[r["document"] for r in news])
    top = test.query(query_texts=["Where can I park my bike?"], n_results=3)
    print(f"{space:<7} {top['ids'][0]}  distances {np.round(top['distances'][0], 3)}")
    client.delete_collection(f"test-{space}")

cosine  ['news-10', 'news-05', 'news-09']  distances [0.253 0.555 0.589]


ip      ['news-10', 'news-05', 'news-09']  distances [0.253 0.555 0.589]


l2      ['news-10', 'news-05', 'news-09']  distances [0.506 1.109 1.179]


| Space | Chroma returns | For normalized vectors |
|---|---|---|
| `cosine` | 1 − cosine similarity | 0 to 2 |
| `ip` (inner product) | 1 − dot product | the same numbers as `cosine` |
| `l2` | the **squared** L2 distance | 2 − 2 · cosine, twice the cosine distance |

All three find the same records in the same order, just as Notebook 1 predicted for
normalized vectors. But the numbers differ, and `l2` is not the L2 distance from Notebook 1:
it is its **square**. Two more things to know:

- **Always set the space yourself.** Without it, Chroma picks a default that depends on
  the embedding function (`l2` when there is none).
- **Chroma does not compare the question with every vector.** It keeps an **HNSW** index
  (the `hnsw` in the setting), a structure that finds close vectors without checking all
  of them. Notebook 3 shows how such indexes work.

## 5. Filters

Metadata is where a vector database beats the simple search from Notebook 1. Stage 2
promised this one: search **only the IT FAQ**.

In [8]:
question = "My laptop is broken. What should I do?"

print("All records:")
show(collection.query(query_texts=[question], n_results=3))
print("Only the IT FAQ:")
show(collection.query(query_texts=[question], n_results=3, where={"source": "it_faq.html"}))

All records:
   0.229  it_faq-06     My laptop is slow or broken. What now?
   0.438  handbook-10   8. Equipment and IT security
   0.440  it_faq-08     Are my files backed up?
Only the IT FAQ:


   0.229  it_faq-06     My laptop is slow or broken. What now?
   0.440  it_faq-08     Are my files backed up?
   0.468  it_faq-02     How do I reset my password?


Without a filter, the handbook section about equipment comes second. With
`where={"source": "it_faq.html"}`, Chroma only ranks the FAQ records.

Filters can compare numbers and combine conditions:

In [9]:
print("Announcements from 2026 or later:")
show(
    collection.query(
        query_texts=["What is new for employees?"], n_results=3, where={"year": {"$gte": 2026}}
    )
)

print("IT announcements from 2026:")
show(
    collection.query(
        query_texts=["What did IT announce?"],
        n_results=3,
        where={"$and": [{"team": "IT"}, {"year": 2026}]},
    )
)

Announcements from 2026 or later:
   0.344  news-08       New handbook version
   0.480  news-12       Book your learning days
   0.498  news-09       New VPN app
IT announcements from 2026:
   0.548  news-11       Phishing training
   0.572  news-09       New VPN app


The second query asked for 3 results but got 2: only two records match the filter. A
filter can return fewer than `n_results`, and your code should expect that.

| Operator | Meaning | Example |
|---|---|---|
| `$eq`, `$ne` | equal, not equal | `{"team": "IT"}` is short for `{"team": {"$eq": "IT"}}` |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than (or equal) | `{"year": {"$gte": 2026}}` |
| `$in`, `$nin` | in a list, not in a list | `{"team": {"$in": ["IT", "People"]}}` |
| `$and`, `$or` | combine conditions | `{"$and": [{...}, {...}]}` |

Two more tools:

- `get(where=...)` returns **all** matching records, without a question and without
  ranking. It is a plain database lookup.
- `where_document` filters by the **text** itself, for example records that contain an
  exact word. It is case-sensitive.

In [10]:
facilities = collection.get(where={"team": "Facilities"})
print("Facilities announcements:", facilities["ids"])

vpn = collection.get(where_document={"$contains": "VPN"})
print('Records that contain "VPN":', vpn["ids"])

Facilities announcements: ['news-02', 'news-05', 'news-07', 'news-10']
Records that contain "VPN": ['it_faq-03', 'news-09']


## 6. Update and delete

Documents change. Imagine HR announces a new learning budget. `upsert()` adds a record
when its id is new and replaces it when the id already exists:

In [11]:
collection.upsert(
    ids=["news-13"],
    documents=[
        "Learning budget raised. From January 2027, the learning budget grows from "
        "1,200 to 1,500 EUR per year."
    ],
    metadatas=[
        {
            "source": "announcements",
            "section": "Learning budget raised",
            "team": "People",
            "date": "2026-11-20",
            "year": 2026,
        }
    ],
)
show(collection.query(query_texts=["How much is the learning budget?"], n_results=2))

   0.220  news-13       Learning budget raised
   0.241  handbook-12   10. Learning and growth


Now two records answer the question, and they disagree: the handbook says 1,200 EUR, the
new announcement says 1,500 EUR from 2027. Neither is wrong, they are valid at different
times. This is why the date belongs in the metadata: section 8 puts it into the prompt,
so the model can tell old from new.

`update()` changes an existing record, and Chroma embeds the new text again. `delete()`
removes records by id or by filter:

In [12]:
collection.update(
    ids=["news-13"],
    documents=[
        "Learning budget raised. From January 2027, the learning budget grows from "
        "1,200 to 1,400 EUR per year."
    ],
)
print("After update:", collection.get(ids=["news-13"])["documents"][0])

collection.delete(ids=["news-13"])
print("After delete:", collection.count(), "records")

After update: Learning budget raised. From January 2027, the learning budget grows from 1,200 to 1,400 EUR per year.
After delete: 36 records


You can also delete by filter, for example `collection.delete(where={"year": 2024})`.

## 7. Save to disk

Everything so far lives in memory and is gone when the notebook stops. `PersistentClient`
saves the collection to a folder instead. The usual pattern: open the collection with
`get_or_create_collection()`, and only embed the documents when it is empty.

In [13]:
shutil.rmtree(DB_PATH, ignore_errors=True)  # start fresh every time this notebook runs


def open_collection():
    persistent_client = chromadb.PersistentClient(path=str(DB_PATH))
    return persistent_client.get_or_create_collection(
        name="brightleaf",
        embedding_function=embedding_function,
        configuration={"hnsw": {"space": "cosine"}},
    )


for run in (1, 2):
    start = time.perf_counter()
    saved = open_collection()
    if saved.count() == 0:  # empty: embed and add everything
        saved.add(
            ids=[r["id"] for r in records],
            documents=[r["document"] for r in records],
            metadatas=[r["metadata"] for r in records],
        )
    print(f"Run {run}: {saved.count()} records ready in {time.perf_counter() - start:.2f} s")

print("Files on disk:", sorted(path.name for path in DB_PATH.iterdir()))

Run 1: 36 records ready in 0.42 s
Run 2: 36 records ready in 0.01 s
Files on disk: ['2ab50bbf-3cc2-4449-b29a-a4a2456b58f3', 'chroma.sqlite3']


The first run embedded all records; the second found them and embedded nothing. The
folder holds a small SQLite database (`chroma.sqlite3`) and a folder with the index.
Restart the kernel and run only this cell: the records are still there.

For thousands of documents, this is the difference between minutes and milliseconds
every time your app starts.

## 8. Chroma in a RAG chain

The `langchain-chroma` package wraps a Chroma collection as a LangChain vector store, so
everything from Stage 2 works with it: retrievers, chains, and sources. It can open the
collection we just saved:

In [14]:
vector_store = Chroma(
    collection_name="brightleaf",
    embedding_function=get_embeddings(),  # LangChain's Ollama embeddings, same model
    persist_directory=str(DB_PATH),
    collection_configuration={"hnsw": {"space": "cosine"}},  # used only when creating
)

for doc, distance in vector_store.similarity_search_with_score("What changed with the VPN?", k=3):
    print(f"   {distance:.3f}  {doc.metadata['section']}")

   0.321  New VPN app
   0.364  How do I connect to the VPN?
   0.460  What is the guest Wi-Fi password?


The same records with the same distances as in section 4, because both sides use the same
embedding model. Pass `collection_configuration` when LangChain **creates** a collection:
its default distance is also `l2`.

Now a RAG chain. Two things are new compared with Stage 2: the retriever can take a Chroma
filter, and the context shows each record's date, so the model can tell old from new.

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions from Brightleaf employees, using only the context below. "
            "The answer may be a yes or a no, or may need a small step of reasoning from the "
            "context. Only if the context does not cover the question, say: I don't know."
            "\n\nContext:\n{context}",
        ),
        ("user", "{question}"),
    ]
)
answer_chain = prompt | get_chat_model() | StrOutputParser()


def format_docs(docs) -> str:
    """Put each record's date (if it has one) in front of its text."""
    return "\n\n---\n\n".join(
        f"[{doc.metadata['date']}] {doc.page_content}"
        if "date" in doc.metadata
        else doc.page_content
        for doc in docs
    )


def ask(question: str, where: dict | None = None) -> None:
    search_kwargs = {"k": 3} if where is None else {"k": 3, "filter": where}
    docs = vector_store.as_retriever(search_kwargs=search_kwargs).invoke(question)
    answer = answer_chain.invoke({"context": format_docs(docs), "question": question})
    print(f"Q: {question}\nA: {answer}")
    print("   sources:", " | ".join(doc.metadata["section"] for doc in docs), "\n")


ask("What changed with the VPN, and by when do I have to act?")
ask("What did the IT team announce in 2026?", where={"$and": [{"team": "IT"}, {"year": 2026}]})

Q: What changed with the VPN, and by when do I have to act?
A: The old VPN app is being replaced by a new one.  
You must install the new VPN app from the company software center **before 31 March 2026**. After that date the old app will stop working.
   sources: New VPN app | How do I connect to the VPN? | New handbook version 



Q: What did the IT team announce in 2026?
A: In 2026 the IT team announced two key updates:

1. **Phishing training** – Every employee must complete a 20‑minute phishing training in the learning portal by **31 July 2026**.  
2. **New VPN app** – A new VPN application replaces the old one. Employees need to install it from the software center before **31 March 2026**; after that date the old VPN will stop working.
   sources: Phishing training | New VPN app 



The first answer comes from the new announcement, including its deadline. The second only
sees what the filter allows: the IT announcements from 2026.

## 9. Summary

| Idea | What it means | In code |
|---|---|---|
| Client | Connection to Chroma: in memory or on disk | `chromadb.EphemeralClient()`, `chromadb.PersistentClient(path)` |
| Collection | Named set of records with one embedding function and one distance | `client.get_or_create_collection(name, embedding_function=..., configuration=...)` |
| Record | id + document + embedding + metadata | `collection.add(ids=..., documents=..., metadatas=...)` |
| Query | The closest records to a question, as distances | `collection.query(query_texts=[...], n_results=3)` |
| Distance space | `cosine` = 1 − cosine, `ip` = 1 − dot, `l2` = squared L2 | `configuration={"hnsw": {"space": "cosine"}}` |
| Metadata filter | Rank only the records that match | `where={"year": {"$gte": 2026}}` |
| Text filter | Records whose text contains a word | `where_document={"$contains": "VPN"}` |
| Update / delete | Change or remove records; changed text is embedded again | `upsert`, `update`, `delete` |
| LangChain | Chroma as a vector store and retriever | `Chroma(...).as_retriever(search_kwargs={"filter": ...})` |

**Remember:** put everything you may want to filter by (source, date, team) into the
metadata when you add the documents. Adding it later means updating every record.

➡️ **Next:** [Notebook 3 — FAISS and fast indexes](03_faiss_and_ann_indexes.ipynb)